In [1]:
"""
Full BioC PMC cleaning pipeline: heading-stack merge -> content-aware
classification (TABLE/FIG/SUPPL) -> canonical schema -> cleaned jsonl
-> corpus metrics + charts.

Every classification decision below was made by pasting real examples
from the corpus, not guessed from field names. Where we haven't seen
real data yet (SUPPL subsection taxonomy), it's an HONEST placeholder
that logs unknowns instead of silently absorbing them into a junk-drawer
bucket. Don't trust "supplementary_other" as a final answer -- it's a
"come back once you've looked" flag, not a design decision.
"""

import json
import re
from dataclasses import dataclass, field
from collections import Counter, defaultdict

import matplotlib.pyplot as plt


# ---------------------------------------------------------------------------
# Section types dropped BEFORE tree-building even starts.
# These are whole-passage drops based on section_type alone -- confirmed
# boilerplate/noise with no internal structure worth preserving.
# ---------------------------------------------------------------------------

DROP_SECTION_TYPES = {
    "REF",          # 103,149 -- bibliography, no retrieval value
    "COMP_INT",     # 3,758  -- competing interests boilerplate
    "ACK_FUND",     # 907    -- acknowledgments/funding boilerplate
    "AUTH_CONT",    # 2,703  -- author contribution statements
    "REVIEW_INFO",  # 449    -- peer review metadata (dates etc.)
    "ABBR",         # 8,121  -- abbreviation glossary, not a retrievable claim
}

# TABLE, FIG, SUPPL are deliberately NOT in this set -- they need
# content-aware handling below, not a blanket keep/drop.


# ---------------------------------------------------------------------------
# Heading-stack merge (unchanged core logic from earlier)
# ---------------------------------------------------------------------------

HEADING_TYPE_RE = re.compile(r"^(abstract_)?title_(\d+)$")


@dataclass
class Section:
    heading_text: str | None
    level: int
    section_type: str
    paragraphs: list[dict] = field(default_factory=list)  # {"type": ptype, "text": text}
    children: list["Section"] = field(default_factory=list)
    synthetic: bool = False

    def merged_text(self) -> str:
        return "\n\n".join(p["text"] for p in self.paragraphs)


def heading_level(passage_type: str) -> int | None:
    m = HEADING_TYPE_RE.match(passage_type)
    return int(m.group(2)) if m else None


def build_section_tree(passages: list[dict]) -> Section:
    """
    Same stack logic as before. NOTE: this now runs on TABLE/FIG/SUPPL
    passages too (previously only body sections) -- SUPPL in particular
    has real title_1/title_2/title_3 nesting we didn't know about until
    we saw the type breakdown, so it needs proper tree reconstruction,
    not flat per-passage handling.
    """
    root = Section(heading_text=None, level=0, section_type="ROOT", synthetic=True)
    stack: list[Section] = [root]
    passages = sorted(passages, key=lambda p: p.get("offset", 0))

    for p in passages:
        infons = p.get("infons", {})
        section_type = infons.get("section_type", "UNKNOWN")
        ptype = infons.get("type", "")
        text = p.get("text", "").strip()

        if section_type in DROP_SECTION_TYPES or not text:
            continue

        lvl = heading_level(ptype)

        if lvl is not None:
            while len(stack) - 1 > lvl - 1:
                stack.pop()
            while len(stack) - 1 < lvl - 1:
                synth = Section(None, len(stack), section_type, synthetic=True)
                stack[-1].children.append(synth)
                stack.append(synth)
            new_section = Section(text, lvl, section_type, synthetic=False)
            stack[-1].children.append(new_section)
            stack.append(new_section)

        else:
            if ptype == "front":
                title_section = Section(None, 1, "TITLE", synthetic=False)
                title_section.paragraphs.append({"type": ptype, "text": text})
                root.children.append(title_section)
                continue

            if len(stack) == 1:
                synth = Section(None, 1, section_type, synthetic=True)
                root.children.append(synth)
                stack.append(synth)

            elif stack[-1].section_type != section_type:
                while len(stack) > 1:
                    stack.pop()
                synth = Section(None, 1, section_type, synthetic=True)
                stack[-1].children.append(synth)
                stack.append(synth)

            # store ptype alongside text now -- TABLE/FIG classifiers need
            # to see individual passage types (table_caption vs table),
            # not just the merged blob
            stack[-1].paragraphs.append({"type": ptype, "text": text})

    return root


def flatten_sections(node: Section, path: list[str] | None = None) -> list[dict]:
    path = path or []
    out = []
    current_path = path + ([node.heading_text] if node.heading_text else [])
    if node.paragraphs:
        out.append({
            "heading_path": [h for h in current_path if h],
            "section_type": node.section_type,
            "level": node.level,
            "paragraphs": node.paragraphs,   # keep per-passage type+text
            "text": node.merged_text(),
            "synthetic": node.synthetic,
        })
    for child in node.children:
        out.extend(flatten_sections(child, current_path))
    return out


# ---------------------------------------------------------------------------
# Canonical section mapping -- body sections (simple section_type lookup)
# ---------------------------------------------------------------------------

CANONICAL_SECTION_MAP = {
    "TITLE": "title",
    "ABSTRACT": "abstract",
    "INTRO": "introduction",
    "METHODS": "methods",
    "RESULTS": "results",
    "DISCUSS": "discussion",
    "CONCL": "conclusion",
    "CASE": "case_report",
    "KEYWORD": "keywords",
    "APPENDIX": "appendix",
}

UNMAPPED_SECTION_TYPES: Counter = Counter()


# ---------------------------------------------------------------------------
# TABLE classification -- driven by real `type` breakdown, evidence-based
# ---------------------------------------------------------------------------

TABLE_KEEP_TYPE_MAP = {
    "table_caption": "table_caption",
    "table_title_caption": "table_caption",
    "table_footnote": "table_footnote",
    "table_foot": "table_footnote",
}
TABLE_DROP_TYPES = {"table"}  # confirmed raw tab-dump, e.g. "Biomarker\tTarget\t..."

TABLE_JUNK_RE = re.compile(r"^\s*references?\s*:?\.?\s*$", re.IGNORECASE)


def classify_table_passage(ptype: str, text: str) -> str | None:
    if TABLE_JUNK_RE.match(text):
        return None
    if ptype in TABLE_DROP_TYPES:
        return None
    if ptype in TABLE_KEEP_TYPE_MAP:
        return TABLE_KEEP_TYPE_MAP[ptype]
    # rare strays: paragraph/title_1/title_2 under TABLE (39 total) --
    # low volume, fold into caption bucket rather than special-case
    return "table_caption"


# ---------------------------------------------------------------------------
# FIG classification -- same evidence-based approach
# ---------------------------------------------------------------------------

FIG_KEEP_TYPE_MAP = {
    "fig_caption": "figure_caption",
    "fig_title_caption": "figure_caption",
    "fig_footnote": "figure_footnote",
}
FIG_BARE_LABEL_RE = re.compile(r"^\s*fig(ure)?\.?\s*\d+\.?\s*$", re.IGNORECASE)

UNMAPPED_FIG_TYPES: Counter = Counter()


def classify_fig_passage(ptype: str, text: str) -> str | None:
    if ptype in FIG_KEEP_TYPE_MAP:
        return FIG_KEEP_TYPE_MAP[ptype]
    if ptype == "fig":
        if FIG_BARE_LABEL_RE.match(text.strip()):
            return None  # bare "Fig. 1" label, no content
        return "figure_caption"  # real descriptive text (e.g. boxplot description)
    if ptype in ("paragraph", "title_1"):
        return "figure_caption"
    UNMAPPED_FIG_TYPES[ptype] += 1
    return "figure_other"


# ---------------------------------------------------------------------------
# SUPPL classification -- POST-MERGE, by heading path (real content exists,
# not just boilerplate labels -- confirmed by paragraph/footnote volume)
# ---------------------------------------------------------------------------

SUPPL_DROP_HEADING_PATTERNS = re.compile(
    r"ethics statement|supplementary material$|conflict of interest|"
    r"author contribution",
    re.IGNORECASE,
)

UNMAPPED_SUPPL_HEADINGS: Counter = Counter()


# Boilerplate pointer headings -- confirmed junk, no inline retrievable
# content, just references to external files/repos/registries.
SUPPL_JUNK_HEADING_PATTERNS = re.compile(
    r"^\s*$|"
    r"supplementary (information|material|data|files?)s?:?$|"
    r"supplemental (information|material|data)s?$|"
    r"supporting information$|"
    r"full text availability$|"
    r"code availability$|"
    r"data (and (code|software))? ?(sharing|availabilit(y|ty)|of availability)"
    r"( statement:?)?$|"
    r"source data$|"
    r"extended data$|"
    r"orcid\s*id s?$|"
    r"resource availability( (lead|technical) contact| "
    r"(materials|data and code) availability)?$|"
    r"declarations?$|"
    r"declaration of (the use of )?generative ai.*$|"
    r"consent (statement|for publication|to participate)$|"
    r"ethic(al|s) (statement|considerations?|approval.*)$|"
    r"informed consent$|"
    r"approval and registration number$|"
    r"studies on human and/or animal statement$|"
    r"research data transparency and availability$|"
    r".*consortium affiliations$"
    r"orcid\s+(.*)|"
    r".*data.*$|"
    r".*supplementary.*$",
    re.IGNORECASE,
)

def classify_suppl_section(heading_path: list[str]) -> str | None:
    label = " ".join(heading_path).strip() if heading_path else ""
    print(f"SUPPL heading path: {heading_path} -> label: {label!r}")

    if SUPPL_JUNK_HEADING_PATTERNS.match(label):
        return None  # confirmed boilerplate pointer, no inline content

    if SUPPL_DROP_HEADING_PATTERNS.search(label):  # ethics/conflict/author-contrib
        return None

    if re.search(r"data availability", label, re.IGNORECASE):
        return "data_availability"

    # Numbered, specific, multi-word headings that survive the junk filter
    # above are very likely misfiled body content (confirmed: the "6.
    # Therapeutic Applications..." review subsections). Keep these as real
    # body text rather than losing them to a section_type mistag.
    if re.match(r"^\d+(\.\d+)*\.?\s+\S", label):
        return "body_misfiled_as_suppl"

    # Anything else genuinely unrecognized -- still log it, don't guess
    UNMAPPED_SUPPL_HEADINGS[label] += 1
    return "supplementary_other"


# ---------------------------------------------------------------------------
# Master classifier -- dispatches to the right evidence-based handler
# ---------------------------------------------------------------------------

def classify_section(flat_section: dict) -> list[dict]:
    """
    Takes one flattened (merged) section and returns a list of final
    section records ready for the schema. Usually returns 0 or 1 records,
    but TABLE/FIG sections can explode into multiple records since each
    passage within needs its OWN classification (a table section might mix
    a caption passage with a raw-dump passage that gets dropped).
    """
    section_type = flat_section["section_type"]
    heading_path = flat_section["heading_path"]
    records = []

    if section_type == "TABLE":
        for para in flat_section["paragraphs"]:
            canonical = classify_table_passage(para["type"], para["text"])
            if canonical:
                records.append(_make_record(heading_path, section_type, canonical,
                                              flat_section["level"], para["text"],
                                              flat_section["synthetic"]))
        return records

    if section_type == "FIG":
        for para in flat_section["paragraphs"]:
            canonical = classify_fig_passage(para["type"], para["text"])
            if canonical:
                records.append(_make_record(heading_path, section_type, canonical,
                                              flat_section["level"], para["text"],
                                              flat_section["synthetic"]))
        return records

    if section_type == "SUPPL":
        canonical = classify_suppl_section(heading_path)
        if canonical:
            records.append(_make_record(heading_path, section_type, canonical,
                                          flat_section["level"], flat_section["text"],
                                          flat_section["synthetic"]))
        return records

    # everything else: simple lookup
    canonical = CANONICAL_SECTION_MAP.get(section_type)
    if canonical is None:
        UNMAPPED_SECTION_TYPES[section_type] += 1
        canonical = "other"
    records.append(_make_record(heading_path, section_type, canonical,
                                  flat_section["level"], flat_section["text"],
                                  flat_section["synthetic"]))
    return records


def _make_record(heading_path, section_type, canonical, level, text, synthetic) -> dict:
    return {
        "heading_path": heading_path,
        "section_type": section_type,
        "canonical_section_type": canonical,
        "level": level,
        "text": text,
        "n_words": len(text.split()),
        "n_chars": len(text),
        "synthetic": synthetic,
    }


# ---------------------------------------------------------------------------
# Document-level processing
# ---------------------------------------------------------------------------

def process_document(doc: dict) -> dict:
    doc_id = doc.get("id")
    passages = doc.get("passages", [])

    meta = {"doi": None, "pmc_id": doc_id, "pmid": None, "year": None,
             "kwd": [], "title": None}
    for p in passages:
        infons = p.get("infons", {})
        if infons.get("type") == "front":
            meta["doi"] = infons.get("article-id_doi")
            meta["pmc_id"] = infons.get("article-id_pmc", doc_id)
            meta["pmid"] = infons.get("article-id_pmid")
            meta["year"] = infons.get("year")
            raw_kwd = infons.get("kwd", "")
            meta["kwd"] = [k.strip() for k in re.split(r"[;,]", raw_kwd) if k.strip()] \
                if ";" in raw_kwd or "," in raw_kwd else raw_kwd.split()
            meta["title"] = p.get("text", "").strip()
            break

    tree = build_section_tree(passages)
    flat_sections = flatten_sections(tree)

    sections = []
    for flat in flat_sections:
        sections.extend(classify_section(flat))

    for i, s in enumerate(sections):
        s["section_id"] = f"{doc_id}_s{i}"

    n_synthetic = sum(1 for s in sections if s["synthetic"])

    return {
        "doc_id": doc_id,
        "pmc_id": meta["pmc_id"],
        "doi": meta["doi"],
        "pmid": meta["pmid"],
        "year": meta["year"],
        "kwd": meta["kwd"],
        "title": meta["title"],
        "n_sections": len(sections),
        "n_synthetic_sections": n_synthetic,
        "sections": sections,
    }


# ---------------------------------------------------------------------------
# I/O
# ---------------------------------------------------------------------------

def load_raw_documents(path: str):
    """
    Handles two jsonl line shapes defensively -- verify against your actual
    file's first line before trusting this blindly on all 1300.
    """
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if "passages" in obj:
                yield obj
            elif "documents" in obj[0]:
                yield from obj[0]["documents"]
            else:
                raise ValueError(f"Unrecognized jsonl line shape, keys={list(obj.keys())}")


def save_cleaned_jsonl(cleaned_docs: list[dict], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for doc in cleaned_docs:
            f.write(json.dumps(doc, ensure_ascii=False) + "\n")


# ---------------------------------------------------------------------------
# Corpus metrics + charts
# ---------------------------------------------------------------------------

def compute_corpus_metrics(cleaned_docs: list[dict]) -> dict:
    canonical_counts = Counter()
    synthetic_by_canonical = defaultdict(lambda: [0, 0])  # [synthetic, total]
    doc_word_totals = []
    section_word_counts = []
    synthetic_per_doc = []
    year_counts = Counter()

    for doc in cleaned_docs:
        doc_words = 0
        year_counts[doc.get("year") or "unknown"] += 1
        synthetic_per_doc.append(doc["n_synthetic_sections"])

        for s in doc["sections"]:
            canonical_counts[s["canonical_section_type"]] += 1
            section_word_counts.append(s["n_words"])
            doc_words += s["n_words"]

            bucket = synthetic_by_canonical[s["canonical_section_type"]]
            bucket[1] += 1
            if s["synthetic"]:
                bucket[0] += 1

        doc_word_totals.append(doc_words)

    return {
        "canonical_counts": canonical_counts,
        "synthetic_by_canonical": synthetic_by_canonical,
        "doc_word_totals": doc_word_totals,
        "section_word_counts": section_word_counts,
        "synthetic_per_doc": synthetic_per_doc,
        "year_counts": year_counts,
        "unmapped_section_types": UNMAPPED_SECTION_TYPES,
        "unmapped_fig_types": UNMAPPED_FIG_TYPES,
        "unmapped_suppl_headings": UNMAPPED_SUPPL_HEADINGS,
    }


def plot_corpus_metrics(metrics: dict, output_path: str = "corpus_metrics.png") -> None:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    ax = axes[0, 0]
    labels, counts = zip(*metrics["canonical_counts"].most_common())
    ax.bar(labels, counts, color="steelblue")
    ax.set_title("Section count by canonical type")
    ax.tick_params(axis="x", rotation=60)

    ax = axes[0, 1]
    cats = list(metrics["synthetic_by_canonical"].keys())
    synth_vals = [metrics["synthetic_by_canonical"][c][0] for c in cats]
    real_vals = [metrics["synthetic_by_canonical"][c][1] - metrics["synthetic_by_canonical"][c][0] for c in cats]
    ax.bar(cats, real_vals, label="real heading", color="seagreen")
    ax.bar(cats, synth_vals, bottom=real_vals, label="synthetic", color="indianred")
    ax.set_title("Synthetic vs. real sections by type")
    ax.tick_params(axis="x", rotation=60)
    ax.legend()

    ax = axes[0, 2]
    max_synth = max(metrics["synthetic_per_doc"], default=0)
    ax.hist(metrics["synthetic_per_doc"], bins=range(0, max_synth + 2), color="darkorange")
    ax.set_title("Synthetic sections per document")
    ax.set_xlabel("# synthetic sections")
    ax.set_ylabel("# documents")

    ax = axes[1, 0]
    ax.hist(metrics["section_word_counts"], bins=40, color="mediumpurple")
    ax.set_title("Section length (words)")
    ax.set_xlabel("words per section")

    ax = axes[1, 1]
    ax.hist(metrics["doc_word_totals"], bins=30, color="teal")
    ax.set_title("Document length (total words)")
    ax.set_xlabel("words per document")

    ax = axes[1, 2]
    years = sorted(metrics["year_counts"].keys(), key=lambda y: (y == "unknown", y))
    ax.bar(years, [metrics["year_counts"][y] for y in years], color="slategray")
    ax.set_title("Documents by publication year")
    ax.tick_params(axis="x", rotation=60)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close(fig)

    if metrics["unmapped_section_types"]:
        print("WARNING -- unmapped section_types (bucketed as 'other'):")
        for st, count in metrics["unmapped_section_types"].most_common():
            print(f"  {st}: {count}")

    if metrics["unmapped_fig_types"]:
        print("WARNING -- unmapped FIG passage types (bucketed as 'figure_other'):")
        for st, count in metrics["unmapped_fig_types"].most_common():
            print(f"  {st}: {count}")

    if metrics["unmapped_suppl_headings"]:
        print("\nSUPPL headings currently bucketed as 'supplementary_other' -- "
              "REVIEW THESE and split into real categories, don't leave as-is:")
        for heading, count in metrics["unmapped_suppl_headings"].most_common(30):
            print(f"  [{count:4d}] {heading!r}")

In [2]:
RAW_PATH = "/home/razor/Documents/alzheimers-rag/data/raw/pmc_corpus.jsonl"
CLEAN_PATH = "/home/razor/Documents/alzheimers-rag/data/processed/pmc_alzheimers_cleaned.jsonl"
CHART_PATH = "/home/razor/Documents/alzheimers-rag/data/charts/corpus_metrics.png"


cleaned = [process_document(doc) for doc in load_raw_documents(RAW_PATH)]
save_cleaned_jsonl(cleaned, CLEAN_PATH)

metrics = compute_corpus_metrics(cleaned)
plot_corpus_metrics(metrics, CHART_PATH)

print(f"Processed {len(cleaned)} documents -> {CLEAN_PATH}")
print(f"Charts saved -> {CHART_PATH}")

SUPPL heading path: ['Data availability statement'] -> label: 'Data availability statement'
SUPPL heading path: ['Ethics statement'] -> label: 'Ethics statement'
SUPPL heading path: ['Supplementary material'] -> label: 'Supplementary material'
SUPPL heading path: ['Data availability statement'] -> label: 'Data availability statement'
SUPPL heading path: ['Ethics statement'] -> label: 'Ethics statement'
SUPPL heading path: ['Supplementary material'] -> label: 'Supplementary material'
SUPPL heading path: ['Data availability'] -> label: 'Data availability'
SUPPL heading path: ['Full Text Availability'] -> label: 'Full Text Availability'
SUPPL heading path: ['Full Text Availability'] -> label: 'Full Text Availability'
SUPPL heading path: ['Full Text Availability'] -> label: 'Full Text Availability'
SUPPL heading path: ['Data availability statement'] -> label: 'Data availability statement'
SUPPL heading path: ['Ethics statement'] -> label: 'Ethics statement'
SUPPL heading path: ['Supplement

Match found: <re.Match object; span=(0, 8), match='ORCID iD'>
